#### **Data analysis for bacteria gene annotation**

In [2]:
#pip install biopython
#!pip install orfipy
#!pip install intervaltree
#!pip install line-profiler
#!pip install memory-profiler
#!pip install filelock

In [3]:
import pandas as pd

#### **Assembly summary**

In [4]:
# loading
assembly_summary  = pd.read_csv('../data/ref/assembly_summary.txt', sep='\t', skiprows=1, low_memory=False, usecols=['refseq_category', 'release_type', 'annotation_date', 'assembly_level', 'organism_name', 'ftp_path', '#assembly_accession', 'asm_name'])

# filtering
annotated_summary = assembly_summary[~assembly_summary['annotation_date'].isna()]
annotated_summary = annotated_summary[annotated_summary['assembly_level'] == "Complete Genome"]
annotated_summary = annotated_summary[annotated_summary['refseq_category'] == "reference genome"]
annotated_summary = annotated_summary.reset_index(drop=True)

In [5]:
annotated_summary.head(5)

,#assembly_accession,refseq_category,organism_name,assembly_level,release_type,asm_name,ftp_path,annotation_date
0,GCA_001027285.1,reference genome,Archangium gephyra,Complete Genome,Major,ASM102728v1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...,2015-06-03
1,GCA_001189295.1,reference genome,Chondromyces crocatus,Complete Genome,Major,ASM118929v1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...,2015-07-30
2,GCA_022870945.1,reference genome,Vitreoscilla stercoraria,Complete Genome,Major,ASM2287094v1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...,2022-01-25
3,GCA_002222655.1,reference genome,Vitreoscilla filiformis,Complete Genome,Major,ASM222265v1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...,2017-07-20
4,GCA_002355975.1,reference genome,Lysobacter enzymogenes,Complete Genome,Major,ASM235597v1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...,2019-04-18


In [6]:
annotated_summary["refseq_category"].unique()

array(['reference genome'], dtype=object)

In [7]:
annotated_summary["release_type"].unique()

array(['Major', 'Minor'], dtype=object)

In [8]:
annotated_summary.shape

(5745, 8)

In [9]:
annotated_summary['organism_name'].value_counts()

organism_name
Entomobacter blattae                       2
Rickettsia tillamookensis                  2
Nodularia sphaerocarpa UHCC 0038           2
Croceimicrobium hydrocarbonivorans         2
Hymenobacter sediminicola                  2
                                          ..
Oceanidesulfovibrio marinus                1
Salinispora tropica CNB-440                1
Janthinobacterium svalbardensis            1
Bifidobacterium adolescentis ATCC 15703    1
Chryseobacterium gambrini                  1
Name: count, Length: 4823, dtype: int64

In [10]:
print(f"Unique organisme : {len(annotated_summary['organism_name'].unique())}")

Unique organisme : 4823


#### **Data Extracting & Labeling**

In [11]:
%load_ext cython
%load_ext line_profiler
%load_ext memory_profiler

In [12]:
import subprocess, gc, warnings, os, re, json, gzip, random, logging
import pandas as pd
import numpy as np
from tqdm import tqdm
from Bio import SeqIO
from itertools import groupby
import multiprocessing as mp
import concurrent.futures
import orfipy_core as oc
from sklearn.utils import shuffle
from filelock import FileLock

logging.basicConfig(
    filename='logger.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

warnings.simplefilter(action='ignore', category=FutureWarning)
gc.collect()

16

In [13]:
#%%cython -a
def extract_dna(sequences, sequence_id, start_pos=None, end_pos=None):
    for record in sequences:
        if record.id == sequence_id:
            sequence = str(record.seq)
            if start_pos is not None and end_pos is not None:
                return sequence[start_pos - 1:end_pos]
            return sequence
    return None

In [14]:
def parse_orfs(input_data):
    orfs_pos, orfs_neg = [], []
    orf_data = input_data.strip().split('>')[1:]
    
    for entry in orf_data:
        lines = entry.splitlines()
        header = lines[0]
        sequence = ''.join(lines[1:])
        
        # extract start, end, strand, frame, length, start codon, end codon from the header using regex
        match = re.search(r'(\d+)-(\d+)\]\((\+|-)\) type:complete length:(\d+) frame:(\d+) start:(\w{3}) stop:(\w{3})', header)
        if match:
            start = int(match.group(1))
            end = int(match.group(2))
            strand = match.group(3)
            length = int(match.group(4))
            frame = int(match.group(5))
            start_codon = match.group(6)
            end_codon = match.group(7)

            if strand == "+":
                orfs_pos.append({
                    'start': start,
                    'end': end,
                    'strand': strand,
                    'length': length,
                    'frame': frame,
                    'start_codon': start_codon,
                    'end_codon': end_codon,
                    'sequence': sequence
                })
            elif strand == "-":
                orfs_neg.append({
                    'start': start,
                    'end': end,
                    'strand': strand,
                    'length': length,
                    'frame': frame,
                    'start_codon': start_codon,
                    'end_codon': end_codon,
                    'sequence': sequence
                })
    
    return orfs_pos, orfs_neg

In [15]:
#%%cython -a
def cds_annotator(cds_df, orfs_dict, strand, max_seq_len=510):
    results = []
    if strand == "+":
        positive_orfs   = orfs_dict
        for index, cds_row in cds_df.iterrows():
            seqid   = cds_row["seqid"]
            cds_end = cds_row["end"]
            matching_orfs  = [orf for orf in positive_orfs if orf['end'] == cds_end]
            positive_orfs  = [orf for orf in positive_orfs if orf['end'] != cds_end]
            if matching_orfs:
                longest_orf = max(matching_orfs, key=lambda d: d['length'])
                results.append({"sequence": longest_orf["sequence"][-max_seq_len:], "label": 1})

        # remaining orfs
        grouped = groupby(sorted(positive_orfs, key=lambda x: x['end']), key=lambda x: x['end'])
        for end_value, group in grouped:
            orf = max(group, key=lambda x: x['length'])
            if orf['length'] > 100:
                results.append({"sequence": orf["sequence"][-max_seq_len:], "label": 0})
    elif strand == "-":
        negative_orfs   = orfs_dict
        for index, cds_row in cds_df.iterrows():
            seqid   = cds_row["seqid"]
            cds_start = cds_row["start"]
            matching_orfs  = [orf for orf in negative_orfs if orf['start'] == cds_start]
            negative_orfs  = [orf for orf in negative_orfs if orf['start'] != cds_start]
            if matching_orfs:
                longest_orf = max(matching_orfs, key=lambda d: d['length'])
                results.append({"sequence": longest_orf["sequence"][:max_seq_len], "label": 1})

        # remaining orfs
        grouped = groupby(sorted(negative_orfs, key=lambda x: x['start']), key=lambda x: x['start'])
        for start_value, group in grouped:
            orf = max(group, key=lambda x: x['length'])
            if orf['length'] > 100:
                results.append({"sequence": orf["sequence"][:max_seq_len], "label": 0})
    return results

In [16]:
def data_balancer(df, id_):
    positive_class = df[df['label'] == 1]
    negative_class = df[df['label'] == 0]
    positive_lengths = positive_class['sequence'].apply(len)
    positive_length_distribution = positive_lengths.tolist() 

    # filter
    negative_class_balanced = []
    for length in positive_length_distribution:
        possible_negatives = negative_class[negative_class['sequence'].apply(len) == length]
        if not possible_negatives.empty:
            negative_class_balanced.append(possible_negatives.sample(n=1, random_state=42))

    if negative_class_balanced:
        negative_class_balanced = pd.concat(negative_class_balanced, ignore_index=True)
    else:
        print("Warning: No matching negative samples found. Returning only positive class data.")
        logging.warning(f'No matching negative samples found. Returning only positive class data.: {id_}')
        negative_class_balanced = negative_class # pd.DataFrame(columns=df.columns)  # Empty DataFrame

    balanced_df = pd.concat([positive_class, negative_class_balanced], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    return balanced_df

In [16]:
storage_folder = "../../GP4Bacteria/Data/RawCompletGenonme/"
etl_folder_raw = "../data/temp/ORF/"
etl_folder     = "../data/cds/BacteriaCDS-ORFSeq2Class-v2/"
json_path      = "../data/cds/BacteriaCDS-ORFSeq2Class-v2/data_summary.json"
os.makedirs(etl_folder_raw, exist_ok=True)
os.makedirs(etl_folder+"database/", exist_ok=True)
os.makedirs(etl_folder+"balanced/", exist_ok=True)
if not os.path.exists(json_path):
    with open(json_path, 'w') as file:
        json.dump({}, file)

def extractor(row_index, row):
    # init
    debug = False
    folder_name = f"{row['#assembly_accession']}_{row['asm_name']}"
    folder_path = os.path.join(storage_folder, folder_name)
    gff_path = os.path.join(folder_path, "genomic.gff.gz")
    fna_path = os.path.join(folder_path, "genomic.fna.gz")

    if not os.path.exists(folder_path):
        logging.warning(f'Missing bacteria: {folder_name}')
        return None
    
    # cds
    columns = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]
    with gzip.open(gff_path, 'rt') as f:
        lines = [line.strip() for line in f if not line.startswith('#')]
    gff_data = [line.split('\t') for line in lines]
    gff_df = pd.DataFrame(gff_data, columns=columns)
    gff_df["start"] = pd.to_numeric(gff_df["start"])
    gff_df["start"] = gff_df["start"] - 1
    gff_df["end"]   = pd.to_numeric(gff_df["end"])
    gff_df["end"]   = gff_df["end"]
    cds_df          = gff_df[gff_df["type"] == "CDS"]
    if debug:
        cds_df.to_csv("temp/cds.csv")
    cds_pos_df = cds_df[cds_df["strand"] == "+"]
    cds_neg_df = cds_df[cds_df["strand"] == "-"]
    
    # fna
    if debug:
        with gzip.open(fna_path, 'rb') as genome_file:
            with open("temp/genome.fna", 'wb') as file:
                file.write(genome_file.read())
    with gzip.open(fna_path, 'rt') as f:
        fna_data = list(SeqIO.parse(f, "fasta"))

    # orf 
    df_list = []
    for record in fna_data:
        seq      = str(record.seq)#[:1000000]
        seq_rc   = str(record.seq.reverse_complement())#[:1000000]
        bacteria = record.description.split(",")[0]
        orfs     = oc.start_search(seq,                                                #seq
                                seq_rc,                                                #seq_rc
                                bacteria,                                              #seqname
                                10,                                                    #minlen
                                10000000,                                              #maxlen
                                'b',                                                   #strand
                                ['TTG', 'CTG', 'ATG', 'GTG'],                          #starts
                                ['TAA', 'TAG', 'TGA'],                                 #stops
                                '1',                                                   #table
                                True,                                                  #include_stop
                                False,                                                 #partial3
                                False,                                                 #partial5
                                False,                                                 #between_stops
                                True,                                                  #nested
                                [False,False,True,False,False]                         #[bed12, bed, dna, rna, pep]    
                )
        if debug:
            with open('temp/orfs_dna.fna', 'w') as file:
                file.write(orfs[2])
        orfs_pos_dict, orfs_neg_dict = parse_orfs(orfs[2])

        multiprocessing = False
        if not multiprocessing:
            # processing (07:21)
            df1 = cds_annotator(cds_pos_df, orfs_pos_dict, "+")
            df2 = cds_annotator(cds_neg_df, orfs_neg_dict, "-")
        else:
            # multiprocessing  (09:13)
            num_processes = 100
            chunks = np.array_split(cds_pos_df, num_processes)
            with mp.Pool(processes=num_processes) as pool:
                results = pool.starmap(cds_annotator, [(chunk, orfs_pos_dict, "+") for chunk in chunks])
            df1 = [item for sublist in results for item in sublist]
            
            chunks = np.array_split(cds_neg_df, num_processes)
            with mp.Pool(processes=num_processes) as pool:
                results = pool.starmap(cds_annotator, [(chunk, orfs_neg_dict, "-") for chunk in chunks])
            df2 = [item for sublist in results for item in sublist]
        
        df_list.extend(df1)
        df_list.extend(df2)

    # save df & update data_summary
    df = pd.DataFrame(df_list)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    df_file_path = etl_folder+f"database/bacteria_{row_index+1}.csv"
    df.to_csv(df_file_path, index=False)
    df_balanced = data_balancer(df, folder_name)
    df_balanced_file_path = etl_folder+f"balanced/bacteria_{row_index+1}.csv"
    df_balanced.to_csv(df_balanced_file_path, index=False)

    # only one process writes at a time
    with FileLock(json_path + ".lock"):  
        with open(json_path, 'r') as file:
            data_summary = json.load(file)
        data_summary[folder_name] = {"bacteria": row["organism_name"], "path": df_file_path}
        with open(json_path, 'w') as file:
            json.dump(data_summary, file, indent=4)

# -----------------------------------------------------------------------------------
annotated_summary_ = annotated_summary.iloc[3350:]
#with concurrent.futures.ProcessPoolExecutor(max_workers=50) as executor:
#    futures = []
#    for row_index, row in annotated_summary_.iterrows():
#        future = executor.submit(extractor, row_index, row)
#        futures.append(future)
#    
#    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Processing rows"):
#        future.result()

max_workers = 100
with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = {}
    for row_index, row in annotated_summary_.iloc[:max_workers].iterrows():
        future = executor.submit(extractor, row_index, row)
        futures[future] = (row_index, row)

    with tqdm(total=len(annotated_summary_), desc="Processing rows") as pbar:
        for row_index, row in annotated_summary_.iloc[max_workers:].iterrows():
            # wait for at least one task to finish before submitting a new one
            done_future = next(concurrent.futures.as_completed(futures))
            done_future.result()
            pbar.update(1)
            del futures[done_future]
            future = executor.submit(extractor, row_index, row)
            futures[future] = (row_index, row)

        # Wait for remaining futures to finish
        for done_future in concurrent.futures.as_completed(futures):
            done_future.result()
            pbar.update(1)


# -----------------------------------------------------------------------------------
#for row_index, row in tqdm(annotated_summary_.iterrows(), desc="Parsing seq: "):
#    extractor(row_index, row)
#    #if row_index == 0:
#    break 

Processing rows:   1%|▏                     | 15/2395 [00:38<1:19:22,  2.00s/it]

Processing rows:  34%|███████▊               | 811/2395 [41:50<47:14,  1.79s/it]

Processing rows:  34%|███████▊               | 815/2395 [41:57<36:06,  1.37s/it]

Processing rows:  73%|██████████████▌     | 1750/2395 [1:32:09<34:48,  3.24s/it]

Processing rows:  73%|██████████████▌     | 1751/2395 [1:32:10<25:27,  2.37s/it]

Processing rows: 100%|████████████████████| 2395/2395 [2:10:55<00:00,  3.28s/it]


In [17]:
os.cpu_count() 

56

In [18]:
--stop--

SyntaxError: invalid syntax (2229127232.py, line 1)

### **Balance DATA**

NB: Too much duplication in first balanced data

In [72]:
def data_balancer_v2(df, id_):
    positive_class = df[df['label'] == 1]
    negative_class = df[df['label'] == 0].copy()
    positive_lengths = positive_class['sequence'].apply(len)
    positive_length_distribution = positive_lengths.tolist()

    negative_class_balanced = []
    for length in positive_length_distribution:
        possible_negatives = negative_class[negative_class['sequence'].apply(len) == length]
        if not possible_negatives.empty:
            selected_sample = possible_negatives.sample(n=1, random_state=42)
            negative_class_balanced.append(selected_sample)
            
            # deL selected sample from negative_class to prevent duplicates
            negative_class = negative_class.drop(selected_sample.index)

    if negative_class_balanced:
        negative_class_balanced = pd.concat(negative_class_balanced, ignore_index=True)
    else:
        print("Warning: No matching negative samples found. Returning only positive class data.")
        logging.warning(f'No matching negative samples found. Returning only positive class data.: {id_}')
        negative_class_balanced = negative_class # pd.DataFrame(columns=df.columns)  # Empty DataFrame
        
    # check balance
    num_positives = len(positive_class)
    num_negatives = len(negative_class_balanced)

    if num_negatives < num_positives:
        missing_negatives = num_positives - num_negatives
        if len(negative_class) > 0:
            additional_negatives = negative_class.sample(n=min(missing_negatives, len(negative_class)), random_state=42)
            negative_class_balanced = pd.concat([negative_class_balanced, additional_negatives], ignore_index=True)
    
    # shuffle
    balanced_df = pd.concat([positive_class, negative_class_balanced], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    return balanced_df

In [63]:
etl_folder = "../data/cds/BacteriaCDS-ORFSeq2Class-v2/"
init_df_path = os.path.join(etl_folder, 'database')
output_folder = os.path.join(etl_folder, "balanced_v2")
os.makedirs(output_folder, exist_ok=True)


"""
for file in tqdm(os.listdir(init_df_path), desc="Balancing data: "):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(init_df_path, file))
        
        # balance
        df_balanced = data_balancer_v2(df, file)
        df_balanced_file_path = etl_folder+f"balanced_v2/{file}"
        df_balanced.to_csv(df_balanced_file_path, index=False)
    #break
"""

def balance_and_save(file):
    try:
        if file.endswith('.csv'):
            file_path = os.path.join(init_df_path, file)
            df = pd.read_csv(file_path)
            
            # Balance data
            df_balanced = data_balancer_v2(df, file)
            df_balanced.to_csv(os.path.join(output_folder, file), index=False)
        
        return file, "Success"
    except Exception as e:
        return file, f"Error: {e}"

max_workers = min(100, os.cpu_count())
csv_files = [file for file in os.listdir(init_df_path) if file.endswith('.csv')]
with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(balance_and_save, file): file for file in csv_files}

    with tqdm(total=len(csv_files), desc="Balancing data") as pbar:
        for future in concurrent.futures.as_completed(futures):
            file, status = future.result()
            pbar.update(1)
            if "Error" in status:
                print(f"⚠️ Error processing {file}: {status}")

Balancing data:  36%|███▌      | 2052/5742 [11:48<24:44,  2.48it/s]  

⚠️ Error processing bacteria_4262.csv: Error: cannot concatenate object of type '<class 'list'>'; only Series and DataFrame objs are valid


Balancing data:  37%|███▋      | 2122/5742 [12:06<12:13,  4.93it/s]

⚠️ Error processing bacteria_5199.csv: Error: cannot concatenate object of type '<class 'list'>'; only Series and DataFrame objs are valid


Balancing data:  41%|████      | 2355/5742 [13:35<15:21,  3.68it/s]

...

Balancing data:  46%|████▌     | 2629/5742 [15:09<15:09,  3.42it/s]

⚠️ Error processing bacteria_5198.csv: Error: cannot concatenate object of type '<class 'list'>'; only Series and DataFrame objs are valid


Balancing data:  82%|████████▏ | 4716/5742 [27:04<04:55,  3.48it/s]

⚠️ Error processing bacteria_4261.csv: Error: cannot concatenate object of type '<class 'list'>'; only Series and DataFrame objs are valid


Balancing data: 100%|██████████| 5742/5742 [33:43<00:00,  2.84it/s]


In [73]:

for file in tqdm(['bacteria_4262.csv', 'bacteria_5199.csv', 'bacteria_3455.csv', 'bacteria_5198.csv', 'bacteria_4261.csv'], desc="Balancing data: "):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(init_df_path, file))
        
        # balance
        df_balanced = data_balancer_v2(df, file)
        df_balanced_file_path = etl_folder+f"balanced_v2/{file}"
        df_balanced.to_csv(df_balanced_file_path, index=False)

Balancing data:   0%|          | 0/5 [00:00<?, ?it/s]

Balancing data: 100%|██████████| 5/5 [00:00<00:00, 10.63it/s]

In [60]:
# END

In [56]:
df['label'].value_counts()

label
0    1662
1     227
Name: count, dtype: int64

In [57]:
df_balanced.shape
df_balanced['label'].value_counts()

label
0    227
1    227
Name: count, dtype: int64

In [58]:
# Count exact duplicates (same sequence and label)
duplicate_rows = df_balanced.duplicated(subset=['sequence', 'label'], keep=False)
num_duplicate_rows = duplicate_rows.sum()

print(f"Total duplicate rows in df2 (same sequence and label): {num_duplicate_rows}")

Total duplicate rows in df2 (same sequence and label): 0
